# Trying society emulations

#### Referring to these projects:

[OASIS](https://github.com/camel-ai/oasis),
[MiroFish-Offline](https://github.com/nikmcfly/MiroFish-Offline),
[miroclaw](https://github.com/justinfinnn/miroclaw).

Those run thousands of agents on a graph database and a GPU. This is the same idea
at the size of one laptop: personas react to a post and to each other, before it goes
out. A simulation only talks in memory, no plugin is involved, so nothing it says can
reach a platform.

In [1]:
import os
from amoorion import Amoorion

# Examples stay off your real setup: see Amoorion.init
Amoorion.init()


PosixPath('/tmp/amoorion-05ezbezh')

In [2]:

amo = Amoorion()

personas = [
    {
        "name": "ada", 
        "bio": "Follows every release.", 
        "stance": "supportive"
    },
    {
        "name": "bo", 
        "bio": "Has been burned before.", 
        "stance": "critical"
    },
    {
        "name": "cy", 
        "bio": "Just arrived.", 
        "stance": "neutral"
    },
    {
        "name": "di", 
        "bio": "Reads, never writes.", 
        "stance": "silent"
    },
]

the_matrix_movie = amo.simulate(
    topic="moving to our own server", 
    personas=personas, 
    rounds=2, 
    run="trial"
)

for message in the_matrix_movie:
    print(
        f"{message['author']['handle']:4}",
        message['text']
    )

ada  moving to our own server? Yes, finally.
bo   @ada I doubt moving to our own server is what is needed.
cy   @bo What does moving to our own server change, in practice?
ada  @cy moving to our own server? Yes, finally.
bo   @ada I doubt moving to our own server is what is needed.
cy   @bo What does moving to our own server change, in practice?


Without a model these lines are scripted by stance: the same every time. They show
the mechanics and predict nothing. What was said stays in the inbox like anything
else, so rules and searches work on it:

In [3]:
[
    message["text"] for message in amo.search(
        text="doubt", 
        platform="society:trial"
    )
][:1]

['@ada I doubt moving to our own server is what is needed.']

With `llm=True` each persona asks a language model. Here that is a fake server:
point `llm/url` in the vault at Ollama, and the words become real.

In [4]:
import json, threading

from http.server import BaseHTTPRequestHandler, HTTPServer


class FakeModel(BaseHTTPRequestHandler):

    """Speaks the chat completions API, as Ollama or a LiteLLM proxy would."""

    def do_POST(self):

        asked = json.loads(
            self.rfile.read(
                int(
                    self.headers["Content-Length"]
                )
            )
        )

        said = asked["messages"][-1]["content"].splitlines()[0][:50]

        answer = {
            "choices": [
                {
                    "message": {
                        "content": f"(a model's words about: {said})"
                    }
                }
            ]
        }

        self.send_response(200)

        self.end_headers()

        self.wfile.write(
            json.dumps(answer).encode()
        )

    def log_message(self, *args):
        pass


model_server = HTTPServer(
    ("127.0.0.1", 0), 
    FakeModel
)

threading.Thread(
    target=model_server.serve_forever, 
    daemon=True
).start()

os.environ["AMOORION_SECRET_LLM_URL"] = f"http://127.0.0.1:{model_server.server_port}/v1"

os.environ["AMOORION_SECRET_LLM_MODEL"] = "fake"

In [5]:
ultimate_dating_simulator = amo.simulate(
    topic="moving to our own server", 
    personas=personas[:2], 
    rounds=1, 
    run="by-model", 
    llm=True
)

for message in ultimate_dating_simulator:
    print(
        f"{message['author']['handle']:4}",
        message['text']
    )

ada  (a model's words about: The topic: moving to our own server)
bo   (a model's words about: The topic: moving to our own server)


In Python, anything callable can think, a model of your own included:

In [6]:
from amoorion.society import Persona, Society


def shout(persona, topic, feed):
    return f"{topic.upper()}!" if len(feed) < 3 else None


[
    message.text for message in Society(
        [
            Persona(name="e"), 
            Persona(name="f")
        ], 
        think=shout
    ).simulate("news")
]

['NEWS!', 'NEWS!', 'NEWS!']